In [11]:
import numpy as np
from IPython.display import display, Markdown, Latex

def load_data(file_path):
  try:
    with open(file_path, 'r') as f:
        # Читаем все строки и убираем пустые
        lines = [line.strip() for line in f.readlines() if line.strip()]
        
    # 1. Размерность задачи
    # m - количество ГИА, n - количество СЗИ
    m, n = list(map(int, lines[0].split()))
    
    # 2. Матрица A (Цены СЗИ для каждой ГИА)
    # Ожидаем m строк, в каждой по n чисел
    matrix_a_data = []
    for i in range(1, m + 1):
        matrix_a_data.append(list(map(float, lines[i].split())))
    A = np.array(matrix_a_data)
    
    # 3. Вектор b (Бюджеты для каждой ГИА)
    line_idx = m + 1
    b = np.array(list(map(float, lines[line_idx].split())))
    
    # 4. Вектор c (Время взлома)
    line_idx += 1
    c = np.array(list(map(float, lines[line_idx].split())))
    
    # 5. Вектор d (Время реагирования)
    line_idx += 1
    d = np.array(list(map(float, lines[line_idx].split())))
    
    # 6. Коэффициент лямбда (приоритет)
    line_idx += 1
    lam = float(lines[line_idx])

    # Предварительная подготовка
    # Сортируем индексы по убыванию d_j 
    indices = np.argsort(d)[::-1]
    
    return {
        "m": m,
        "n": n,
        "A": A[:, indices], # Столбцы матрицы переставляем так же
        "b": b,
        "c": c[indices],
        "d": d[indices],
        "lambda": lam,
        "original_indices": indices # Чтобы знать, какой индекс был в начале
    }

  except Exception as e:
      print(f"Произошла ошибка при чтении: {e}")
      return None

def display_latex_math_model(data):
    if not data:
        return
    
    # 1. Параметры размерности
    # ГИА - группы информационных активов, СЗИ - средства защиты
    display(Markdown(f"* Количество ГИА: $m = {data['m']}$\n"
                     f"* Количество СЗИ: $n = {data['n']}$\n"
                     f"* Коэффициент приоритета: $\\lambda = {data['lambda']}$"))

    # 2. Векторы эффективности
    # c_j - время взлома, d_j - время реакции [cite: 68, 69]
    c_vals = ", ".join([f"{val:.3f}" for val in data['c']])
    d_vals = ", ".join([f"{val:.3f}" for val in data['d']])
    
    display(Markdown("**Векторы эффективности (дефаззифицированные):**"))
    display(Latex(rf"c = ({c_vals})"))
    display(Latex(rf"d = ({d_vals})"))

    # 3. Матрица ограничений (Ax <= b)
    matrix_rows = []
    for row in data['A']:
        matrix_rows.append(" & ".join([f"{val:.0f}" for val in row]))
    matrix_str = r" \\ ".join(matrix_rows)
    
    vector_x = r" \\ ".join([f"x_{{{i+1}}}" for i in range(data['n'])])
    vector_b = r" \\ ".join([f"{val:.0f}" for val in data['b']])
    
    display(Markdown(f"**Система финансовых ограничений:**"))
    latex_system = (
        r"\begin{pmatrix} " + matrix_str + r" \end{pmatrix} "
        r"\cdot \begin{pmatrix} " + vector_x + r" \end{pmatrix} "
        r"\le \begin{pmatrix} " + vector_b + r" \end{pmatrix}"
    )
    display(Latex(f"$$ {latex_system} $$"))

data = load_data('input.txt')
display_latex_math_model(data)

* Количество ГИА: $m = 3$
* Количество СЗИ: $n = 5$
* Коэффициент приоритета: $\lambda = 0.5$

**Векторы эффективности (дефаззифицированные):**

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

**Система финансовых ограничений:**

<IPython.core.display.Latex object>

In [6]:
import pulp

def solve_knapsack_pulp(c, A, b):
    """
    Вспомогательная функция для решения одномерной или многомерной 
    задачи о ранце с аддитивным критерием.
    """
    n = len(c)
    m = len(A)
    
    # Инициализируем задачу максимизации
    prob = pulp.LpProblem("Knapsack_Problem", pulp.LpMaximize)
    
    # Определяем бинарные переменные x_j (1 если СЗИ куплено, 0 если нет)
    # Согласно статье, x_j - булевы переменные
    x = [pulp.LpVariable(f"x_{j}", cat=pulp.LpBinary) for j in range(n)]
    
    # Целевая функция: максимизируем время взлома (аддитивный критерий)
    prob += pulp.lpSum([c[j] * x[j] for j in range(n)])
    
    # Финансовые ограничения Ax <= b
    for i in range(m):
        prob += pulp.lpSum([A[i][j] * x[j] for j in range(n)]) <= b[i]
    
    # Решаем задачу тихим солвером
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Проверяем статус и возвращаем вектор решений
    if pulp.LpStatus[prob.status] == 'Optimal':
        results = []
        for j in range(n):
            val = pulp.value(x[j])
            # Pylance доволен
            if isinstance(val, (float, int)):
                results.append(int(round(val)))
            else:
                results.append(0)
        return results
    else:
        return None

def algorithm_step_1(data):
    print("Шага 1: Решение аддитивной задачи")
    
    # Решаем задачу для критерия F1(x)
    x0 = solve_knapsack_pulp(data['c'], data['A'], data['b'])
    
    if x0 is None:
        print("Решение не найдено. Проверьте ограничения.")
        return None
    
    # Множество V для хранения векторов-кандидатов
    V = [x0]
    
    print(f"Вектор x^0 найден: {x0}")
    
    # Если лямбда = 1, то x^0 и есть оптимальное решение 
    if data['lambda'] == 1.0:
        print("Коэффициент lambda = 1. Алгоритм завершен.")
        return x0, V
    
    return x0, V

result = algorithm_step_1(data)

if result is not None:
    x0, V = result
else:
    print("Ошибка. Не найдено начальное решение")

Шага 1: Решение аддитивной задачи
Вектор x^0 найден: [0, 1, 0, 1, 1]


In [7]:
def algorithm_step_2(x0):
    # x0 - это список или numpy массив из 0 и 1
    # Ищем все индексы, где значение равно 1
    indices_with_one = [i for i, val in enumerate(x0) if val == 1]
    
    if not indices_with_one:
        print("Внимание: в x0 не выбрано ни одно СЗИ.")
        return 0
    
    # Находим максимальный индекс j0 - 1
    j0_python = max(indices_with_one)
    
    # j0
    j0_math = j0_python + 1
    
    print(f"Шаг 2:")
    print(f"  - Последнее выбранное СЗИ имеет индекс: {j0_math} (внутренний индекс: {j0_python}) ")
    print(f"  - На следующем шаге будет решено {j0_math - 1} вспомогательных задач ")
    
    return j0_math

# Пример использования:
j0 = algorithm_step_2(x0)

Шаг 2:
  - Последнее выбранное СЗИ имеет индекс: 5 (внутренний индекс: 4) 
  - На следующем шаге будет решено 4 вспомогательных задач 


In [8]:
def algorithm_step_3(data, j0, V):
    print(f"\nШага 3: Решение {j0 - 1} вспомогательных задач")
    
    n = data['n']
    m = data['m']
    c = data['c']
    d = data['d']
    A = data['A']
    b = data['b']
    lam = data['lambda']
    
    # Цикл по s от 1 до j0 - 1
    for s in range(1, j0):
        # Количество СЗИ, которые мы рассматриваем в текущей задаче: j0 - s 
        num_vars = j0 - s
        
        # Инициализируем задачу максимизации
        prob = pulp.LpProblem(f"Knapsack_s_{s}", pulp.LpMaximize)
        
        # Создаем переменные только для оставшихся СЗИ
        x = [pulp.LpVariable(f"x_{j}", cat=pulp.LpBinary) for j in range(num_vars)]
        
        # Формируем новую целевую функцию
        # 1. Слагаемые для всех элементов, кроме последнего
        objective = pulp.lpSum([lam * c[j] * x[j] for j in range(num_vars - 1)])
        
        # 2. Слагаемое для последнего рассматриваемого элемента (индекс num_vars - 1)
        # В него "зашивается" и время взлома (c), и время реакции (d)
        last_idx = num_vars - 1
        last_term = (lam * c[last_idx] + (1 - lam) * d[last_idx]) * x[last_idx]
        
        # Добавляем всю функцию в солвер
        prob += objective + last_term
        
        # Финансовые ограничения
        # Используем усеченную матрицу A_s 
        for i in range(m):
            prob += pulp.lpSum([A[i][j] * x[j] for j in range(num_vars)]) <= b[i]
            
        # Запускаем солвер
        prob.solve(pulp.PULP_CBC_CMD(msg=False))
        
        # Если решение найдено
        if pulp.LpStatus[prob.status] == 'Optimal':
            xs_partial = []
            for j in range(num_vars):
                # Получаем значение переменной через pulp.value
                val = pulp.value(x[j])
                
                # Pylance доволен
                if isinstance(val, (int, float)):
                    clean_val = float(val)
                else:
                    clean_val = 0.0
                
                # Округляем и превращаем в int (0 или 1)
                xs_partial.append(int(round(clean_val)))
            
            # Дополняем вектор отсутствующими компонентами (нулями) справа
            xs_full = xs_partial + [0] * (n - num_vars)
            
            # Сохраняем n-мерный булев вектор x^s в множество V
            V.append(xs_full)
            print(f"  [s={s}] Вектор x^{s} найден: {xs_full}")
        else:
            print(f"  [s={s}] Решение не найдено, пропускаем.")
            
    print(f"Шаг 3 завершен. Множество V содержит {len(V)} кандидатов.")
    return V

V = algorithm_step_3(data, j0, V)


Шага 3: Решение 4 вспомогательных задач
  [s=1] Вектор x^1 найден: [0, 1, 1, 1, 0]
  [s=2] Вектор x^2 найден: [0, 1, 1, 0, 0]
  [s=3] Вектор x^3 найден: [0, 1, 0, 0, 0]
  [s=4] Вектор x^4 найден: [1, 0, 0, 0, 0]
Шаг 3 завершен. Множество V содержит 5 кандидатов.


In [9]:
from tabulate import tabulate

def calculate_F(x, c, d, lam):
    # F1 (Аддитивный критерий): сумма c_j для всех выбранных СЗИ
    F1 = sum(c[j] * x[j] for j in range(len(x)))
    
    # F2 (Максиминный критерий): минимальное d_j среди выбранных СЗИ
    selected_d = [d[j] for j in range(len(x)) if x[j] == 1]
    
    # Если не выбрано ни одного СЗИ (на всякий случай), реакция равна 0
    F2 = min(selected_d) if selected_d else 0.0
    
    # Итоговая функция свертки
    F = lam * F1 + (1 - lam) * F2
    return F, F1, F2

def algorithm_step_4_pretty(data, V):
    print("\n" + "="*80)
    print(f"{'Шаг 4: выбор оптимального решения из множества V':^80}")
    print("="*80)
    
    results_table = []
    best_x = None
    best_F = -1.0
    lam = data['lambda']
    
    # Перебираем всех кандидатов из множества V
    for idx, x in enumerate(V):
        # Расчет критериев
        f1_val = sum(data['c'][j] * x[j] for j in range(len(x)))
        
        # Находим минимальное d_j среди выбранных СЗИ (x_j=1)
        selected_d = [data['d'][j] for j in range(len(x)) if x[j] == 1]
        f2_val = min(selected_d) if selected_d else 0.0
        
        # Итоговая свертка
        f_total = lam * f1_val + (1 - lam) * f2_val
        
        # Сохраняем данные для таблицы
        vector_str = str(x)
        results_table.append([f"x^{idx}", vector_str, f"{f1_val:.2f}", f"{f2_val:.2f}", f"{f_total:.4f}"])
        
        # Поиск максимума
        if f_total > best_F:
            best_F = f_total
            best_x = x
            
    # Вывод таблицы
    headers = ["Кандидат", "Вектор решения (x)", "F1 (Взлом)", "F2 (Реакция)", "F (Свертка)"]
    print(tabulate(results_table, headers=headers, tablefmt="grid"))
    
    print("\n" + "*"*80)
    print(f"Оптимальный вектор: {best_x}")
    print(f"Максимальное значение функции: {best_F:.4f}")
    
    # Перевод в исходные номера СЗИ для ЛПР
    original_nums = []
    if best_x is not None:
        original_nums = [
            int(data['original_indices'][i] + 1) 
            for i, val in enumerate(best_x) 
            if val == 1
        ]
    original_nums.sort()
    print(f"Рекомендуемый набор СЗИ: {original_nums}")
    print("*"*80 + "\n")
    
    return best_x, best_F

best_x, best_F = algorithm_step_4_pretty(data, V)


                Шаг 4: выбор оптимального решения из множества V                
+------------+----------------------+--------------+----------------+---------------+
| Кандидат   | Вектор решения (x)   |   F1 (Взлом) |   F2 (Реакция) |   F (Свертка) |
+============+======================+==============+================+===============+
| x^0        | [0, 1, 0, 1, 1]      |          141 |             10 |          75.5 |
+------------+----------------------+--------------+----------------+---------------+
| x^1        | [0, 1, 1, 1, 0]      |          141 |             12 |          76.5 |
+------------+----------------------+--------------+----------------+---------------+
| x^2        | [0, 1, 1, 0, 0]      |          106 |             14 |          60   |
+------------+----------------------+--------------+----------------+---------------+
| x^3        | [0, 1, 0, 0, 0]      |           76 |             20 |          48   |
+------------+----------------------+--------------+------